[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alinespereira/SME0211/blob/main/notebooks/02_grafica.ipynb)

# A gráfica: carteiras de pedidos do artigo

Exemplos de uso do pacote `printing_company`: os dados do artigo *Energy consumption optimization in a printing company*, o modelo relaxado, os objetivos de tempo e de energia e as tabelas do comando `printing-company`. A formulação está na documentação (`make docs`).

A célula abaixo instala os pacotes `simplex` e `printing_company` direto do GitHub (no Colab, ou em qualquer ambiente sem o projeto). Rodando localmente, com `uv sync`, ela pode ser pulada.

In [ ]:
!pip install -q git+https://github.com/alinespereira/SME0211

## 1. Os dados do artigo

In [2]:
import pandas as pd

from printing_company.problem import data

pd.DataFrame([m.model_dump() for m in [*data.printers, *data.finishers]])

,id,speed,setup,consumption
0,Print01,10500.0,0.25,40.0
1,Print02,3000.0,0.50,112.0
2,Print03,7800.0,0.33,49.2
3,Print04,12600.0,0.17,24.0
4,Print05,9000.0,0.25,2.2
5,Finish01,7800.0,0.50,40.0
6,Finish03,10800.0,0.75,50.0
7,Finish02,12000.0,0.75,112.0


In [3]:
pd.DataFrame(
    [
        {
            "carteira": p.label,
            "itens": len(p.demands),
            "impressão (m)": p.total,
            "acabamento (m)": p.with_finishing,
        }
        for p in data.portfolios
    ]
)

,carteira,itens,impressão (m),acabamento (m)
0,Portfolio I,10,543000.0,225000.0
1,Portfolio II,20,1163000.0,543000.0
2,Portfolio III,50,2557000.0,1862000.0


## 2. Um exemplo pequeno

Duas impressoras, uma máquina de acabamento e dois itens, dos quais só o
primeiro precisa de acabamento. As variáveis são as horas que cada máquina
fica ligada; as restrições de demanda estão em milhares de metros.

In [4]:
from printing_company.problem.model import (
    Objective,
    build_problem,
    min_capacity,
)
from printing_company.problem.spec import (
    Demand,
    FinishingMachine,
    Portfolio,
    PrintingMachine,
)

portfolio = Portfolio(
    label="Exemplo",
    demands=[
        Demand(label="D01", quantity=15_000, requires_finishing=True),
        Demand(label="D02", quantity=20_000),
    ],
)
printers = [
    PrintingMachine(id="P1", speed=10_500, setup=0.25, consumption=40),
    PrintingMachine(id="P2", speed=3_000, setup=0.5, consumption=112),
]
finishings = [
    FinishingMachine(id="F1", speed=7_800, setup=0.5, consumption=40),
]

problem = build_problem(
    portfolio=portfolio, printers=printers, finishings=finishings
)
print(problem)

min        P1 +   P2 +     F1
s.a.  10.5 P1 + 3 P2          >= 35
                       7.8 F1 >= 15
      P1, P2, F1 >= 0


Sem capacidade, os 35 000 m vão para a impressora mais rápida, ligada por
$35\,000 / 10\,500 \approx 3{,}33$ h, e os 15 000 m de acabamento, para a
única máquina de acabamento, por $15\,000 / 7\,800 \approx 1{,}92$ h:

In [5]:
solution = problem.solve()
(
    dict(zip(problem.variable_names, solution.x.round(4).tolist())),
    round(solution.objective, 4),
)

({'P1': 3.3333, 'P2': 0.0, 'F1': 1.9231}, 5.2564)

Com um turno de 3 horas, a `P1` fica ligada as 3 horas e imprime 31 500 m;
a `P2` imprime o resto. Com 2 horas, as impressoras só dão conta de
27 000 m, e o problema fica inviável.

In [6]:
for capacity in (3, 2):
    s = build_problem(
        portfolio=portfolio,
        printers=printers,
        finishings=finishings,
        capacity=capacity,
    ).solve()
    print(capacity, s.status, None if s.x is None else s.x.round(4))

3 optimal [3.     1.1667 1.9231]
2 infeasible None


Minimizando a energia, os custos são as potências (kW). A `P1` também gasta
menos energia por metro, então a solução é a mesma, com consumo de
210,26 kWh:

In [7]:
problem = build_problem(
    portfolio=portfolio,
    printers=printers,
    finishings=finishings,
    objective=Objective.ENERGY,
)
problem.c, round(problem.solve().objective, 2)

(array([ 40., 112.,  40.]), 210.26)

## 3. As carteiras do artigo

A menor capacidade $T$ (em horas por máquina) que torna cada carteira
viável, e quantos turnos de 10 h isso dá:

In [8]:
import math

from printing_company.settings import settings

for p in data.portfolios:
    t_min = min_capacity(
        portfolio=p, printers=data.printers, finishings=data.finishers
    )
    print(
        f"{p.label:<13} T_min = {t_min:6.2f} h, {math.ceil(t_min / settings.shift_hours)} turnos"
    )

Portfolio I   T_min =  12.66 h, 2 turnos
Portfolio II  T_min =  27.11 h, 3 turnos
Portfolio III T_min =  60.85 h, 7 turnos


Na carteira I, com dois turnos ($T = 20$ h), o objetivo de tempo liga as
máquinas mais rápidas, e o de energia, as que gastam menos energia por
metro:

In [9]:
for objective in Objective:
    problem = build_problem(
        portfolio=data.portfolios[0],
        printers=data.printers,
        finishings=data.finishers,
        capacity=2 * settings.shift_hours,
        objective=objective,
    )
    s = problem.solve()
    on = {
        n: round(t, 2)
        for n, t in zip(problem.variable_names, s.x.tolist())
        if t > 0
    }
    print(
        f"{objective.label}: {s.objective:.2f} {objective.unit}, {s.x.sum():.2f} h"
    )
    print("   ", on)

tempo: 67.75 h, 67.75 h
    {'Print01': 20.0, 'Print04': 20.0, 'Print05': 9.0, 'Finish02': 18.75}
energia: 1993.01 kWh, 71.73 h
    {'Print01': 10.57, 'Print04': 20.0, 'Print05': 20.0, 'Finish01': 1.15, 'Finish03': 20.0}


## 4. Tempo real, energia e custo

`solve_portfolio` escolhe o menor número de turnos, resolve o modelo e
calcula o tempo real (com um setup por máquina ligada), a energia e o custo.
`portfolio_report` monta a mesma tabela que o comando `printing-company`
mostra:

In [10]:
from rich.console import Console

from printing_company.problem.analysis import solve_portfolio
from printing_company.report import portfolio_report, summary_table

console = Console(width=100)
result = solve_portfolio(
    portfolio=data.portfolios[0],
    printers=data.printers,
    finishings=data.finishers,
)
console.print(portfolio_report(result=result))

────────────────────────────────── Portfolio I · objetivo: tempo ───────────────────────────────────
10 itens · impressão: 543.000 m · acabamento: 225.000 m (5 itens)
Horizonte: 2 turnos de 10 h, T = 20 h por máquina
┏━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ Máquina  ┃ v (m/h) ┃ t (h) ┃  uso ┃ τ (h) ┃ produção (m) ┃    kW ┃  E (kWh) ┃
┡━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ Print01  │  10.500 │ 20,00 │ 100% │ 20,25 │      210.000 │  40,0 │   810,00 │
│ Print02  │   3.000 │     — │    — │     — │            — │ 112,0 │        — │
│ Print03  │   7.800 │     — │    — │     — │            — │  49,2 │        — │
│ Print04  │  12.600 │ 20,00 │ 100% │ 20,17 │      252.000 │  24,0 │   484,08 │
│ Print05  │   9.000 │  9,00 │  45% │  9,25 │       81.000 │   2,2 │    20,35 │
├──────────┼─────────┼───────┼──────┼───────┼──────────────┼───────┼──────────┤
│ Finish01 │   7.800 │     — │    — │     — │            — │  40,0 │        — │
│ Finish03 │  10.800 │     — │    — │     — │            — │  50,0 │        — │
│ Finish02 │  12.000 │ 18,75 │  94% │ 19,50 │      225.000 │ 112,0 │ 2.184,00 │
├──────────┼─────────┼───────┼──────┼───────┼──────────────┼───────┼──────────┤
│ Total    │         │ 67,75 │      │ 69,17 │              │       │ 3.498,43 │
└──────────┴─────────┴───────┴──────┴───────┴──────────────┴───────┴──────────┘
t: horas do modelo · uso: t/T · τ: t + setup · E = kW·τ                        
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Ótimo do modelo ┃ Tempo real (h) ┃ Energia (kWh) ┃ Custo (R$) ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│         67,75 h │          69,17 │      3.498,43 │   3.008,65 │
└─────────────────┴────────────────┴───────────────┴────────────┘

O resumo das três carteiras, com os dois objetivos:

In [11]:
results = [
    solve_portfolio(
        portfolio=p,
        printers=data.printers,
        finishings=data.finishers,
        objective=objective,
    )
    for p in data.portfolios
    for objective in Objective
]
console.print(summary_table(results=results))

                                    Resumo                                    
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Carteira      ┃ objetivo ┃         ótimo ┃  τ (h) ┃   E (kWh) ┃ custo (R$) ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Portfolio I   │ tempo    │       67,75 h │  69,17 │  3.498,43 │   3.008,65 │
│ Portfolio I   │ energia  │  1.993,01 kWh │  73,65 │  2.065,14 │   1.776,02 │
├───────────────┼──────────┼───────────────┼────────┼───────────┼────────────┤
│ Portfolio II  │ tempo    │      162,59 h │ 165,09 │  7.607,13 │   6.542,13 │
│ Portfolio II  │ energia  │  5.870,62 kWh │ 175,97 │  5.958,98 │   5.124,72 │
├───────────────┼──────────┼───────────────┼────────┼───────────┼────────────┤
│ Portfolio III │ tempo    │      423,85 h │ 426,85 │ 19.465,85 │  16.740,63 │
│ Portfolio III │ energia  │ 18.116,05 kWh │ 439,41 │ 18.288,42 │  15.728,04 │
└───────────────┴──────────┴───────────────┴────────┴───────────┴────────────┘
ótimo: valor do modelo · τ: tempo real · E: energia                           